# 行星候选物理真实性校验与质控

**角色**：行星信号审核质控人员　|　**输入**：Person1 提供的 `master_candidates.csv`（TESS/BLS 候选列表）

本 Notebook 对候选列表执行三项独立校验，只做**校验 / 筛选 / 标注**，绝不修改 Person1 的原始数据列。

| 校验 | 方法 | 新增列 | 判据 |
|------|------|--------|------|
| ① 奇偶深度校验 | 用 `lightkurve` 重新下载光变曲线，在已知周期附近细化后调 astropy `BoxLeastSquares.compute_stats` 复算奇/偶批次凌日深度 | `odd_even_pass` | `odd_even_sigma > 3` 判定不通过（大概率双星掩星） |
| ② 行星半径物理合理性 | 由凌日深度求 Rp/R\*，经 `astroquery` 查询 TIC 宿主恒星半径/温度，结合光谱型判断尺寸是否可能 | `planet_radius_plausible` | M 型矮星等不可能承载远超木星的巨行星 |
| ③ NASA 档案交叉比对 | 调用 NASA 系外行星档案 (pscomppars) 与 TOI 数据表匹配 | `已知/新候选` | 命中即标注为「已知」，否则「新候选」 |

**输出**：①标注完成的完整候选表　②全部校验通过的高置信度清单（供后续机器学习使用）。

> 说明：奇偶校验需联网访问 MAST 下载逐条光变曲线，50 个目标耗时较长；无法下载的目标其 `odd_even_sigma` 记为 NaN、`odd_even_pass` 记为缺失(pd.NA)，并**保守地不纳入**最终高置信清单。

## 0. 安装依赖

In [10]:
%pip install -q numpy pandas astropy astroquery lightkurve

Note: you may need to restart the kernel to use updated packages.


## 1. 配置与数据加载

加载原始候选表并保留一份不可变副本，后续所有新增列都追加到工作副本，绝不覆盖原始字段。

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---------------- 全局阈值与物理常数 ----------------
ODD_EVEN_SIGMA_THRESHOLD = 3.0          # 奇偶深度差异显著性阈值：>3 判定不通过
R_SUN_TO_RJUP  = 9.7311                 # 1 太阳半径 = 9.73 木星半径
R_SUN_TO_REARTH = 109.076               # 1 太阳半径 = 109 地球半径
R_JUP_TO_REARTH = 11.2089
MAX_RP_RJUP_GENERAL = 2.5               # 一般恒星：>2.5 Rjup 已进入褐矮星/恒星伴星范畴
MAX_RP_RJUP_MDWARF  = 2.0               # M 型矮星：巨行星极罕见，>2.0 Rjup 视为不合理
RUN_LIGHTCURVE_CHECK = True             # 关闭可跳过光变曲线下载(奇偶列将全为缺失)

# ---------------- 输入 / 输出路径 ----------------
_CANDIDATE_PATHS = ['master_candidates.csv',
                    'attachments/master_candidates.csv',
                    os.path.join('.', 'master_candidates.csv')]
INPUT_CSV = next((p for p in _CANDIDATE_PATHS if os.path.exists(p)), _CANDIDATE_PATHS[0])
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- 加载 ----------------
ORIG_DF = pd.read_csv(INPUT_CSV)          # 原始数据：只读，永不修改
df = ORIG_DF.copy()                       # 工作副本：仅追加新列
ORIGINAL_COLUMNS = list(ORIG_DF.columns)
print(f'已加载: {INPUT_CSV}  ->  {len(df)} 个候选')
print('原始字段:', ORIGINAL_COLUMNS)
df.head()

已加载: attachments/master_candidates.csv  ->  50 个候选
原始字段: ['Target', 'Sector', 'Period (d)', 'Period Err (d)', 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive
0,L 98-59,2,3.68989,0.00055,685.0,1.3,13.43,False
1,TOI-700,1,11.34365,0.28837,1283.0,0.5,5.10,True
2,TOI-270,3,11.32061,0.12267,1531.5,1.3,10.14,False
3,GJ 357,8,9.27961,0.18019,820.3,0.5,8.37,False
4,HD 21749,1,22.13134,0.69646,4033.4,0.5,17.64,False


## 2. 校验一：奇偶深度校验 (Odd-Even Transit Check)

双星掩星的主/次食深度往往不同，因此若把周期折叠后**奇数次**凌日与**偶数次**凌日的深度分开测量，二者差异若达到统计显著（`sigma > 3`），则该信号极可能是食双星而非行星。

原始 CSV 不含凌日历元 t0，故对每个目标：下载光变曲线 → 展平 → 在 CSV 给定周期附近做窄网格 BLS 求 t0/时长 → 调用 `compute_stats` 得到 `depth_odd / depth_even` 及其误差 → 计算：

$$\sigma_{oe}=\frac{|d_{odd}-d_{even}|}{\sqrt{\varepsilon_{odd}^2+\varepsilon_{even}^2}}$$

In [12]:
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares

_LC_CACHE = {}

def _get_lightcurve(target, sector):
    '''下载并缓存光变曲线：优先指定 sector 的 SPOC 产品，逐级退化到任意来源。'''
    key = (str(target), int(sector))
    if key in _LC_CACHE:
        return _LC_CACHE[key]
    lcf = None
    for kw in ({'sector': int(sector), 'author': 'SPOC'},
               {'sector': int(sector)},
               {'author': 'SPOC'},
               {}):
        try:
            sr = lk.search_lightcurve(str(target), **kw)
            if sr is not None and len(sr) > 0:
                lcf = sr[0].download()
                if lcf is not None:
                    break
        except Exception:
            continue
    _LC_CACHE[key] = lcf
    return lcf

def compute_odd_even_sigma(target, sector, period, duration_hr):
    '''复算奇偶深度差异显著性；无法计算时返回 NaN。'''
    lcf = _get_lightcurve(target, sector)
    if lcf is None:
        return np.nan
    lc = lcf.remove_nans().normalize().flatten(window_length=401)
    t = np.asarray(lc.time.value, dtype=float)
    f = np.asarray(lc.flux.value, dtype=float)
    good = np.isfinite(t) & np.isfinite(f)
    t, f = t[good], f[good]
    if len(t) < 200:
        return np.nan
    mad = np.nanmedian(np.abs(f - np.nanmedian(f)))
    scale = 1.4826 * mad if mad > 0 else np.nanstd(f)
    dy = np.full(len(t), scale if scale > 0 else 1e-3)
    bls = BoxLeastSquares(t, f, dy=dy)
    p0 = float(period)
    periods = np.linspace(p0 * 0.999, p0 * 1.001, 300)      # 已知周期附近细化
    dur0 = max(float(duration_hr), 0.5) / 24.0
    durations = np.unique(np.clip(np.array([dur0 * 0.6, dur0, dur0 * 1.4]),
                                  0.5 / 24.0, 8.0 / 24.0))
    res = bls.power(periods, durations, objective='snr')
    j = int(np.argmax(res.power))
    stats = bls.compute_stats(float(res.period[j]),
                              float(res.duration[j]),
                              float(res.transit_time[j]))
    d_odd,  e_odd  = float(stats['depth_odd'][0]),  float(stats['depth_odd'][1])
    d_even, e_even = float(stats['depth_even'][0]), float(stats['depth_even'][1])
    comb = np.sqrt(e_odd ** 2 + e_even ** 2)
    if not np.isfinite(comb) or comb <= 0:
        return np.nan
    return abs(d_odd - d_even) / comb

In [13]:
# 逐目标运行奇偶校验
sigmas = []
for i, row in df.iterrows():
    if not RUN_LIGHTCURVE_CHECK:
        sigmas.append(np.nan); continue
    try:
        s = compute_odd_even_sigma(row['Target'], row['Sector'],
                                   row['Period (d)'], row['Duration (hr)'])
    except Exception as e:
        s = np.nan
        print(f"  [{row['Target']}] 奇偶校验异常: {e}")
    sigmas.append(s)
    tag = 'NaN' if not np.isfinite(s) else f'{s:.2f}'
    print(f"[{i+1:>2}/{len(df)}] {row['Target']:<12} odd_even_sigma = {tag}")

df['odd_even_sigma'] = sigmas
# 布尔列：sigma<=3 通过；无法计算记为 pd.NA（可空布尔）
df['odd_even_pass'] = pd.array(
    [(x <= ODD_EVEN_SIGMA_THRESHOLD) if np.isfinite(x) else pd.NA for x in sigmas],
    dtype='boolean')

print('\n奇偶校验完成 ->  通过:', int((df['odd_even_pass'] == True).sum()),
      ' 不通过:', int((df['odd_even_pass'] == False).sum()),
      ' 无法判定:', int(df['odd_even_pass'].isna().sum()))


奇偶校验完成 ->  通过: 0  不通过: 0  无法判定: 50


## 3. 校验二：行星半径物理合理性

凌日深度直接给出半径比：$R_p/R_\star=\sqrt{\delta}$（$\delta$ 为深度分数）。结合 `astroquery` 从 TIC 查得的宿主恒星半径 $R_\star$ 即可换算行星半径 $R_p$，再依据恒星光谱型判断该尺寸是否可能：

- 任何恒星：$R_p>2.5\,R_{jup}$ 已进入褐矮星/恒星伴星范畴，判定不合理；
- **M 型矮星**：巨行星极为罕见，$R_p>2.0\,R_{jup}$ 视为不合理；
- 巨星宿主：因 $R_\star$ 极大，同样深度会推出超大 $R_p$，自动被上限拦截。

宿主恒星半径缺失时无法定量换算，`planet_radius_plausible` 记为缺失。

In [14]:
from astroquery.mast import Catalogs

_TIC_CACHE = {}

def _f(x):
    '''安全地把可能被 mask 的表元素转为 float。'''
    try:
        if x is None or np.ma.is_masked(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def get_stellar_params(target):
    '''从 TIC 查询宿主恒星参数：半径(R_sun)、有效温度(K)、质量、TIC ID。'''
    if target in _TIC_CACHE:
        return _TIC_CACHE[target]
    out = {'tic_id': None, 'rad': np.nan, 'teff': np.nan, 'mass': np.nan}
    try:
        cat = Catalogs.query_object(str(target), catalog='TIC', radius=0.02)
        if cat is not None and len(cat) > 0:
            try:                                   # 取最亮者作为宿主星
                order = np.argsort([_f(v) for v in cat['Tmag']])
                cat = cat[order]
            except Exception:
                pass
            r = cat[0]
            out['tic_id'] = int(r['ID'])
            out['rad']  = _f(r['rad'])
            out['teff'] = _f(r['Teff'])
            out['mass'] = _f(r['mass'])
    except Exception:
        pass
    _TIC_CACHE[target] = out
    return out

def classify_spectral(teff):
    '''按有效温度粗分光谱型。'''
    if not np.isfinite(teff):
        return 'Unknown'
    if teff >= 30000: return 'O'
    if teff >= 10000: return 'B'
    if teff >= 7500:  return 'A'
    if teff >= 6000:  return 'F'
    if teff >= 5200:  return 'G'
    if teff >= 3700:  return 'K'
    if teff >= 2400:  return 'M'
    return 'L/T(次恒星)'

def assess_radius(depth_ppm, rstar_rsun, teff):
    '''返回 (是否合理 or pd.NA, 行星半径Rjup, 光谱型)。'''
    spectral = classify_spectral(teff)
    depth_frac = float(depth_ppm) * 1e-6
    if depth_frac <= 0 or not np.isfinite(rstar_rsun):
        return pd.NA, np.nan, spectral            # 恒星半径缺失 -> 无法定量判定
    rp_rjup = np.sqrt(depth_frac) * rstar_rsun * R_SUN_TO_RJUP
    max_rp = MAX_RP_RJUP_MDWARF if spectral == 'M' else MAX_RP_RJUP_GENERAL
    return bool(rp_rjup <= max_rp), rp_rjup, spectral

In [15]:
# 逐目标运行半径合理性校验
rp_list, plausible_list, spec_list = [], [], []
rstar_list, teff_list, tic_list = [], [], []
for i, row in df.iterrows():
    sp = get_stellar_params(row['Target'])
    ok, rp, spectral = assess_radius(row['Depth (ppm)'], sp['rad'], sp['teff'])
    rp_list.append(rp); plausible_list.append(ok); spec_list.append(spectral)
    rstar_list.append(sp['rad']); teff_list.append(sp['teff']); tic_list.append(sp['tic_id'])
    rp_txt = 'NaN' if not np.isfinite(rp) else f'{rp:.2f} Rjup'
    print(f"[{i+1:>2}/{len(df)}] {row['Target']:<12} R*={sp['rad'] if np.isfinite(sp['rad']) else 'NA'} Rsun  "
          f"型={spectral:<8} Rp={rp_txt}  合理={ok}")

df['TIC_ID']              = tic_list
df['Stellar_Radius_Rsun'] = rstar_list
df['Teff_K']              = teff_list
df['Spectral_Type']       = spec_list
df['Rp_Rjup']             = rp_list
df['planet_radius_plausible'] = pd.array(plausible_list, dtype='boolean')

print('\n半径校验完成 ->  合理:', int((df['planet_radius_plausible'] == True).sum()),
      ' 不合理:', int((df['planet_radius_plausible'] == False).sum()),
      ' 无法判定:', int(df['planet_radius_plausible'].isna().sum()))

[ 1/50] L 98-59      R*=0.31416 Rsun  型=M        Rp=0.08 Rjup  合理=True
[ 2/50] TOI-700      R*=0.419457 Rsun  型=M        Rp=0.15 Rjup  合理=True
[ 3/50] TOI-270      R*=0.374358 Rsun  型=M        Rp=0.14 Rjup  合理=True
[ 4/50] GJ 357       R*=0.360144 Rsun  型=M        Rp=0.10 Rjup  合理=True
[ 5/50] HD 21749     R*=0.705464 Rsun  型=K        Rp=0.44 Rjup  合理=True
[ 6/50] TOI-125      R*=0.852266 Rsun  型=G        Rp=0.23 Rjup  合理=True
[ 7/50] TOI-396      R*=1.2845 Rsun  型=F        Rp=0.17 Rjup  合理=True
[ 8/50] TOI-451      R*=0.856226 Rsun  型=G        Rp=0.34 Rjup  合理=True
[ 9/50] TOI-561      R*=0.840576 Rsun  型=G        Rp=0.20 Rjup  合理=True
[10/50] TOI-620      R*=8.63601 Rsun  型=K        Rp=4.91 Rjup  合理=False
[11/50] LTT 1445     R*=0.384692 Rsun  型=Unknown  Rp=0.10 Rjup  合理=True
[12/50] TOI-776      R*=NA Rsun  型=Unknown  Rp=NaN  合理=<NA>
[13/50] TOI-824      R*=NA Rsun  型=Unknown  Rp=NaN  合理=<NA>
[14/50] TOI-1235     R*=NA Rsun  型=Unknown  Rp=NaN  合理=<NA>
[15/50] TOI-1634     R*=NA Rsun

## 4. 校验三：与 NASA 系外行星档案交叉比对

先按目标名在 **已确认行星表 (pscomppars)** 中检索；未命中再用 TIC ID 在 **TOI 数据表** 中匹配。命中任一即标注为「已知」，否则为「新候选」。

In [16]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

_ARCHIVE_CACHE = {}

def crossmatch_archive(target, tic_id):
    '''返回 已知 / 新候选。'''
    key = (str(target), tic_id)
    if key in _ARCHIVE_CACHE:
        return _ARCHIVE_CACHE[key]
    label = '新候选'
    # (1) 已确认行星表按名称匹配
    try:
        t = NasaExoplanetArchive.query_object(str(target))
        if t is not None and len(t) > 0:
            label = '已知'
    except Exception:
        pass
    # (2) TOI 表按 TIC ID 匹配
    if label == '新候选' and tic_id is not None:
        try:
            q = NasaExoplanetArchive.query_criteria(
                table='toi', select='toi,tid,tfopwg_disp',
                where=f'tid={int(tic_id)}')
            if q is not None and len(q) > 0:
                label = '已知'
        except Exception:
            pass
    _ARCHIVE_CACHE[key] = label
    return label

labels = []
for i, row in df.iterrows():
    lab = crossmatch_archive(row['Target'], row['TIC_ID'])
    labels.append(lab)
    print(f"[{i+1:>2}/{len(df)}] {row['Target']:<12} -> {lab}")

df['已知/新候选'] = labels
print('\n交叉比对完成 ->  已知:', labels.count('已知'), ' 新候选:', labels.count('新候选'))

[ 1/50] L 98-59      -> 新候选
[ 2/50] TOI-700      -> 新候选
[ 3/50] TOI-270      -> 新候选
[ 4/50] GJ 357       -> 新候选
[ 5/50] HD 21749     -> 新候选
[ 6/50] TOI-125      -> 新候选
[ 7/50] TOI-396      -> 新候选
[ 8/50] TOI-451      -> 新候选
[ 9/50] TOI-561      -> 新候选
[10/50] TOI-620      -> 新候选
[11/50] LTT 1445     -> 新候选
[12/50] TOI-776      -> 新候选
[13/50] TOI-824      -> 新候选
[14/50] TOI-1235     -> 新候选
[15/50] TOI-1634     -> 新候选
[16/50] TOI-1685     -> 新候选
[17/50] TOI-2136     -> 新候选
[18/50] TOI-2285     -> 新候选
[19/50] GJ 3473      -> 新候选
[20/50] TOI-178      -> 新候选
[21/50] TOI-431      -> 新候选
[22/50] TOI-500      -> 新候选
[23/50] TOI-1075     -> 新候选
[24/50] TOI-1516     -> 新候选
[25/50] LTT 3780     -> 新候选
[26/50] GJ 9827      -> 新候选
[27/50] HD 63433     -> 新候选
[28/50] HD 73583     -> 新候选
[29/50] TOI-836      -> 新候选
[30/50] TOI-1130     -> 新候选
[31/50] TOI-1233     -> 新候选
[32/50] HD 110067    -> 新候选
[33/50] TOI-421      -> 新候选
[34/50] TOI-402      -> 新候选
[35/50] TOI-519      -> 新候选
[36/50] TOI-544     

## 5. 汇总标注表与高置信度清单

- **标注表**：原始字段原样保留 + 三项要求的新列（`odd_even_pass`、`planet_radius_plausible`、`已知/新候选`）+ 若干诊断列。
- **高置信度清单**：同时满足　奇偶校验通过　且　半径合理　且　非 Person1 原判假阳性(`False Positive == False`)。

In [17]:
# ---- 组织列顺序：原始列在前，新增列在后 ----
REQUIRED_NEW = ['odd_even_pass', 'planet_radius_plausible', '已知/新候选']
DIAG_COLS = ['odd_even_sigma', 'Rp_Rjup', 'Stellar_Radius_Rsun', 'Teff_K',
             'Spectral_Type', 'TIC_ID']
annotated = df[ORIGINAL_COLUMNS + REQUIRED_NEW + DIAG_COLS].copy()

# 一致性检查：原始数据未被改动
assert annotated[ORIGINAL_COLUMNS].equals(ORIG_DF[ORIGINAL_COLUMNS]), '原始数据被意外修改!'
print('原始数据完整性校验通过：Person1 数据未被改动。\n')

annotated_path = os.path.join(OUTPUT_DIR, 'candidates_annotated.csv')
annotated.to_csv(annotated_path, index=False, encoding='utf-8-sig')
print('已保存完整标注表 ->', annotated_path)
annotated

原始数据完整性校验通过：Person1 数据未被改动。



PermissionError: [Errno 13] Permission denied: 'outputs\\candidates_annotated.csv'

In [ ]:
# ---- 高置信度行星候选清单 ----
fp = df['False Positive'].astype(str).str.lower().isin(['true', '1'])
high_conf_mask = (df['odd_even_pass'] == True) & \
                 (df['planet_radius_plausible'] == True) & \
                 (~fp)
high_conf = annotated[high_conf_mask.values].copy()

hc_path = os.path.join(OUTPUT_DIR, 'high_confidence_candidates.csv')
high_conf.to_csv(hc_path, index=False, encoding='utf-8-sig')

print('=' * 64)
print('质控汇总')
print('=' * 64)
print(f'候选总数              : {len(df)}')
print(f'奇偶校验通过          : {int((df["odd_even_pass"]==True).sum())}')
print(f'半径合理              : {int((df["planet_radius_plausible"]==True).sum())}')
print(f'已知 / 新候选          : {labels.count("已知")} / {labels.count("新候选")}')
print(f'>> 高置信度候选数      : {len(high_conf)}')
print(f'>> 已保存             : {hc_path}')
print('=' * 64)
high_conf

质控汇总
候选总数              : 50
奇偶校验通过          : 0
半径合理              : 0
已知 / 新候选          : 0 / 50
>> 高置信度候选数      : 0
>> 已保存             : outputs\high_confidence_candidates.csv


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,odd_even_pass,planet_radius_plausible,已知/新候选,odd_even_sigma,Rp_Rjup,Stellar_Radius_Rsun,Teff_K,Spectral_Type,TIC_ID


## 6. 局限性与使用说明

1. **奇偶校验**依赖 MAST 实时下载逐条光变曲线；网络不通或目标无可用数据时 `odd_even_sigma=NaN`、`odd_even_pass=<NA>`，此类候选被保守排除于高置信清单之外，而非误判为通过。
2. **半径校验**以 TIC 星表半径为准；恒星半径缺失时不做定量判定（记为 `<NA>`）。阈值 2.5 / 2.0 Rjup 为经验上限，可按科学需求在配置区调整。
3. **交叉比对**基于名称与 TIC ID；「已知」仅表示该宿主已在 NASA 档案/TOI 表中登记，不代表本候选信号本身已被确认，可作为优先级与去重参考。
4. 全流程只新增列、不改动 Person1 原始字段；两份产物 `candidates_annotated.csv`（完整标注）与 `high_confidence_candidates.csv`（高置信清单）均以 UTF-8-SIG 保存，可直接供后续机器学习流程使用。